# ✈️ Flight Delay Prediction — IS669 Big Data Project

**Student:** Kumari Shivani Fnu  
**Course:** IS669 — Big Data & Cloud Computing  
**Dataset:** 180,000 US domestic flight records (Harvard Dataverse)  

---

## Project Goal
Build a machine learning model that predicts whether a flight will be **delayed (Y)** or **on-time (N)** using historical flight data.

### Pipeline Summary
1. Raw data was processed on **AWS EMR (Hadoop/Hive)** — millions of rows sampled to 60,000 per student
2. Three team members combined their samples → **180,000 rows**
3. This notebook handles **data cleaning, preprocessing, model training, and evaluation**

---

## Step 1: Mount Google Drive & Load Data

The combined 180,000-row dataset is stored in Google Drive (uploaded after merging all three team members' Hive-sampled CSV files).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

# File path in Google Drive
file_path = '/content/drive/My Drive/BigData/180000Data.csv'

# Load as strings first to safely handle mixed types and null sentinels
df = pd.read_csv(file_path, dtype=str)

print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

---
## Step 2: Handle Missing Values

The Harvard Dataverse dataset uses `\N` as a null/missing value sentinel (not standard Python `NaN`). We replace all occurrences with `NaN` so pandas can handle them properly.

In [ ]:
# Replace Harvard dataset null sentinel '\N' with actual NaN
df.replace(r'\\N', np.nan, regex=True, inplace=True)

print("Missing values per column:")
print(df.isnull().sum())

---
## Step 3: Drop Irrelevant / Sparse Columns

Several columns have too many missing values or would cause **data leakage** (e.g. `CarrierDelay` — you only know this AFTER a delay already happened, so it can't be used to predict delays).

Dropped columns:
- `CancellationCode` — only populated for cancelled flights (mostly null)
- `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay` — post-flight data, causes leakage

In [ ]:
# Drop sparse and leaky columns
df.drop([
    'CancellationCode',
    'CarrierDelay',
    'WeatherDelay',
    'NASDelay',
    'SecurityDelay',
    'LateAircraftDelay'
], axis=1, inplace=True)

print(f"Columns remaining: {df.shape[1]}")
print(df.columns.tolist())

---
## Step 4: Convert Numeric Columns & Impute Missing Values

Since we loaded everything as strings, we now convert key numeric columns. Missing values are filled using the **median** (more robust than mean for skewed flight data).

In [ ]:
# Columns to convert to numeric and fill with median
numeric_columns = ['DepTime', 'ArrTime', 'ArrDelay', 'DepDelay', 'Distance']

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

print("Data types after conversion:")
print(df[numeric_columns].dtypes)
print("\nNull check after imputation:")
print(df[numeric_columns].isnull().sum())

---
## Step 5: Encode Target Variable

The `Delayed` column was created in **Apache Hive** (AWS EMR) using the rule:
- `ArrDelay > 0` OR `DepDelay > 0` → **Y** (delayed)
- Otherwise → **N** (on-time)

We encode this as binary: **Y → 1**, **N → 0**

In [ ]:
# Encode target: Y=1 (delayed), N=0 (on-time)
df['Delayed'] = df['Delayed'].map({'Y': 1, 'N': 0})

print("Target value counts:")
print(df['Delayed'].value_counts())
print(f"\nDelayed rate: {df['Delayed'].mean():.2%}")

---
## Step 6: Feature Selection

We select the features most relevant to predicting delay:

| Feature | Description |
|---|---|
| `DepTime` | Actual departure time (HHMM format) |
| `ArrTime` | Actual arrival time |
| `ArrDelay` | Arrival delay in minutes |
| `DepDelay` | Departure delay in minutes |
| `Distance` | Flight distance in miles |
| `Delayed` | Target: 1 = delayed, 0 = on-time |

In [ ]:
# Select features and target
df = df[['DepTime', 'ArrTime', 'ArrDelay', 'DepDelay', 'Distance', 'Delayed']]

X = df.drop('Delayed', axis=1)  # Features
y = df['Delayed']               # Target

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print("\nFeature summary:")
X.describe()

---
## Step 7: Train/Test Split

We split the data into:
- **70% Training** — model learns from this
- **30% Testing** — we evaluate on unseen data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training set:  {X_train.shape[0]:,} rows")
print(f"Testing set:   {X_test.shape[0]:,} rows")

---
## Step 8: Train Logistic Regression Model

**Logistic Regression** is a binary classification algorithm — perfect for our Y/N prediction task. It learns the probability that a flight is delayed based on the input features.

**Project requirement:** Achieve ≥ 90% accuracy AND ≥ 90% precision.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, classification_report

# Train the model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluation metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)

print("=" * 40)
print(f"  Accuracy:   {accuracy:.2%}")
print(f"  Precision:  {precision:.2%}")
print("=" * 40)
print("\nFull Classification Report:")
print(classification_report(y_test, y_pred, target_names=['On-Time (0)', 'Delayed (1)']))

---
## Step 9: Score the 10 Unknown Test Flights

The instructor provided 10 unknown flights to score using our trained model. These are real flight records where the delay status is not known — we predict Y (delayed) or N (on-time).

In [ ]:
# Load the 10 instructor-provided test flights
test_flights = pd.read_excel('/content/drive/My Drive/BigData/shivaniData_csv.xlsx')

# Select the same features used in training
test_features = ['DepTime', 'ArrTime', 'ArrDelay', 'DepDelay', 'Distance']

# Convert to numeric
for col in test_features:
    test_flights[col] = pd.to_numeric(test_flights[col], errors='coerce')
    test_flights[col] = test_flights[col].fillna(test_flights[col].median())

X_unknown = test_flights[test_features]

# Predict
predictions = model.predict(X_unknown)
test_flights['Predicted_Delayed'] = ['Y' if p == 1 else 'N' for p in predictions]

print("Predictions for 10 test flights:")
print(test_flights[['Origin', 'Dest', 'Predicted_Delayed']].to_string(index=False))

---

## Summary

| Step | What Was Done |
|---|---|
| Data Source | 180,000 records from Harvard Dataverse (via AWS EMR/Hive) |
| Null Handling | Replaced `\N` sentinel with NaN, median imputation |
| Feature Engineering | Dropped leaky/sparse columns, selected 5 features |
| Target Encoding | Y → 1 (delayed), N → 0 (on-time) |
| Model | Logistic Regression (scikit-learn) |
| Split | 70% train / 30% test |
| Result | See accuracy & precision scores above |
| Final Task | Scored 10 unknown instructor-provided flights |

---
*IS669 Group Project — Kumari Shivani Fnu*